# 自前で書く探索ループ（30 行）

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 03 自前で書く——探索ループは 30 行で足りる

**このノートのパス**　`ml-code/05_hpo/03_pure_search_loop.ipynb`

**このノートで確かめること**
ライブラリを使わずランダム探索と逐次半減を書く。
**候補を出す → 学習する → 記録する → 最良を覚える**、それだけであることを確かめ、
**「Optuna 導入前の数字」**を手に入れる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `lightgbm scikit-learn numpy` |
| 実行時間の目安 | 約 3〜5 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q lightgbm scikit-learn numpy

In [3]:
import time, math, random
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import average_precision_score
from lightgbm import LGBMClassifier

SEED = 42
X, y = make_classification(
    n_samples=300_000, n_features=30, n_informative=15, n_redundant=8,
    weights=[0.95, 0.05], class_sep=1.3, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

# 探索は時間がかかるのでサブサンプルで回す（増やせば時間も増える）
N_SUB = 20_000
X_s, y_s = X_tr[:N_SUB], y_tr[:N_SUB]
print("探索に使う件数:", X_s.shape)

探索に使う件数: (20000, 30)


## 探索空間を「引く関数」として定義する（define-by-run の原型）

In [4]:
def sample_params(rng):
    return {
        "n_estimators":  rng.randint(200, 800),
        "num_leaves":    rng.randint(15, 120),
        # 対数一様：桁で効く量は、桁の上で一様に引く
        "learning_rate": math.exp(rng.uniform(math.log(1e-2), math.log(3e-1))),
        "subsample":     rng.choice([0.6, 0.7, 0.8, 0.9, 1.0]),
    }


def objective(params, X, y, cv):
    model = LGBMClassifier(class_weight="balanced", random_state=SEED,
                           subsample_freq=1, verbosity=-1, n_jobs=1, **params)
    return cross_val_score(model, X, y, cv=cv,
                           scoring="average_precision", n_jobs=1).mean()

## 探索ループ本体 — これが「ランダム探索」の全実装である

In [5]:
def random_search(X, y, n_trials=20, seed=SEED):  # 書籍では 60
    rng = random.Random(seed)
    cv  = StratifiedKFold(3, shuffle=True, random_state=seed)
    history, best = [], None
    for i in range(n_trials):
        params = sample_params(rng)
        t0 = time.perf_counter()
        score = objective(params, X, y, cv)
        history.append({"trial": i, "score": score,
                        "sec": time.perf_counter() - t0, **params})
        if best is None or score > best["score"]:
            best = {"trial": i, "score": score, "params": params}
        print(f"  [{i:2d}] {score:.5f}  best={best['score']:.5f}", flush=True)
    return best, history


best, history = random_search(X_s, y_s, n_trials=20)
print("\n最良:", best)

  [ 0] 0.90285  best=0.90285


  [ 1] 0.90136  best=0.90285


  [ 2] 0.90354  best=0.90354


  [ 3] 0.87172  best=0.90354


  [ 4] 0.85305  best=0.90354


...

  [16] 0.90174  best=0.90911


  [17] 0.89456  best=0.90911


  [18] 0.90549  best=0.90911


  [19] 0.90857  best=0.90911



最良: {'trial': 15, 'score': np.float64(0.9091123439771022), 'params': {'n_estimators': 484, 'num_leaves': 73, 'learning_rate': 0.08689264232496482, 'subsample': 0.8}}


In [6]:
import pandas as pd
hist_df = pd.DataFrame(history)
print(hist_df.round(5).to_string(index=False))
# history をそのまま保存すれば、探索履歴の成果物になる
# hist_df.to_csv("random_search_history.csv", index=False)

 trial   score      sec  n_estimators  num_leaves  learning_rate  subsample
     0 0.90285  2.12027           314          18        0.12455        0.7
     1 0.90136  4.01812           428          32        0.12242        1.0
     2 0.90354  4.84740           289          90        0.04200        0.6
     3 0.87172  2.98903           295          42        0.02206        1.0
     4 0.85305  3.83898           227          86        0.01967        1.0
     5 0.90662  8.05093           629          43        0.04608        0.8
     6 0.89934  3.14757           206         112        0.15499        0.9
     7 0.89723  7.24243           548          50        0.01697        0.8
     8 0.88914  2.80292           304          26        0.03641        0.8
     9 0.90562  9.33994           552          92        0.02459        0.6
    10 0.89536 10.94237           670          83        0.01529        0.9
    11 0.87751  4.47217           280          85        0.02711        1.0
    12 0.900

## 逐次半減（Successive Halving）も、外側にループを 1 枚かぶせるだけ

In [7]:
def halving_search(X, y, n_candidates=27, factor=3, seed=SEED, min_frac=1/9):
    rng  = random.Random(seed)
    cv   = StratifiedKFold(3, shuffle=True, random_state=seed)
    cand = [sample_params(rng) for _ in range(n_candidates)]
    frac = min_frac                              # 最初は少ない件数で回す
    while len(cand) > 1:
        n   = max(int(len(X) * frac), 2_000)
        idx = rng.sample(range(len(X)), min(n, len(X)))
        Xs, ys = X[idx], y[idx]
        scored = [(objective(p, Xs, ys, cv), p) for p in cand]
        scored.sort(key=lambda t: -t[0])
        keep = max(len(cand) // factor, 1)       # 上位 1/factor だけ残す
        print(f"  候補 {len(cand):3d} → {keep:3d}  件数 {len(Xs):6d}  "
              f"最良 {scored[0][0]:.5f}", flush=True)
        cand = [p for _, p in scored[:keep]]     # 残りに資源を factor 倍まわす
        frac = min(frac * factor, 1.0)
    return cand[0]


print("逐次半減の最良:", halving_search(X_s, y_s))

  候補  27 →   9  件数   2222  最良 0.71847


  候補   9 →   3  件数   6666  最良 0.86735


  候補   3 →   1  件数  20000  最良 0.90894


逐次半減の最良: {'n_estimators': 638, 'num_leaves': 89, 'learning_rate': 0.03890119793179667, 'subsample': 0.7}


## 読み取りどころ

**ここまで外部ライブラリを 1 つも足していない。**探索フレームワークは魔法ではなく、この 4 行の繰り返しに過ぎない。

**Optuna を入れる前に、この数字を取っておく。**比較の条件は 3 つ揃えれば足りる——
①同じ試行回数 ②同じ乱数種 ③同じ交差検証の分割。

⚠️ **差が出なくても失敗ではない。**TPE は最初の 10 試行がランダムなので、数十試行では差が小さいのが普通である。
**「差が出ない範囲では、自前の 30 行で十分」という結論も、立派な成果物**である。

次は `07_search_comparison.ipynb` で、この自前実装と Optuna を同一条件で比較する。